<a href="https://colab.research.google.com/github/KamaljitGrewal/machine-failure-pyspark/blob/main/machine_failure_pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyspark
!wget -q -O ai4i.zip "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"
!unzip -o -q ai4i.zip
!ls

ai4i2020.csv  ai4i.zip	sample_data


In [2]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("machine-failure").getOrCreate()

csv_path = "ai4i2020.csv"

df = spark.read.csv(csv_path, header=True, inferSchema=True)
print("Rows:", df.count())
df.printSchema()
df.show(5)

Rows: 10000
root
 |-- UDI: integer (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Air temperature [K]: double (nullable = true)
 |-- Process temperature [K]: double (nullable = true)
 |-- Rotational speed [rpm]: integer (nullable = true)
 |-- Torque [Nm]: double (nullable = true)
 |-- Tool wear [min]: integer (nullable = true)
 |-- Machine failure: integer (nullable = true)
 |-- TWF: integer (nullable = true)
 |-- HDF: integer (nullable = true)
 |-- PWF: integer (nullable = true)
 |-- OSF: integer (nullable = true)
 |-- RNF: integer (nullable = true)

+---+----------+----+-------------------+-----------------------+----------------------+-----------+---------------+---------------+---+---+---+---+---+
|UDI|Product ID|Type|Air temperature [K]|Process temperature [K]|Rotational speed [rpm]|Torque [Nm]|Tool wear [min]|Machine failure|TWF|HDF|PWF|OSF|RNF|
+---+----------+----+-------------------+-----------------------+-----------------

In [3]:
df = (
    df.withColumnRenamed("Type", "machine_type")
      .withColumnRenamed("Air temperature [K]", "air_temp")
      .withColumnRenamed("Process temperature [K]", "process_temp")
      .withColumnRenamed("Rotational speed [rpm]", "rot_speed")
      .withColumnRenamed("Torque [Nm]", "torque")
      .withColumnRenamed("Tool wear [min]", "tool_wear")
      .withColumnRenamed("Machine failure", "failure")
)

df = df.select("machine_type", "air_temp", "process_temp",
               "rot_speed", "torque", "tool_wear", "failure")

df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

df = df.dropDuplicates().dropna()
df = df.withColumn("temp_diff", F.col("process_temp") - F.col("air_temp"))
print("Rows after cleaning:", df.count())

+------------+--------+------------+---------+------+---------+-------+
|machine_type|air_temp|process_temp|rot_speed|torque|tool_wear|failure|
+------------+--------+------------+---------+------+---------+-------+
|           0|       0|           0|        0|     0|        0|      0|
+------------+--------+------------+---------+------+---------+-------+

Rows after cleaning: 10000


In [5]:
df.createOrReplaceTempView("machines")

print("Failure rate by machine type:")
spark.sql("""
    SELECT machine_type,
           COUNT(*) AS total_machines,
           SUM(failure) AS failures,
           ROUND(100 * SUM(failure) / COUNT(*), 2) AS failure_rate_pct
    FROM machines
    GROUP BY machine_type
    ORDER BY failure_rate_pct DESC
""").show()

print("Average readings for failed vs working machines:")
spark.sql("""
    SELECT failure,
           ROUND(AVG(tool_wear), 1) AS avg_tool_wear,
           ROUND(AVG(torque), 1) AS avg_torque,
           ROUND(AVG(rot_speed), 1) AS avg_speed,
           ROUND(AVG(temp_diff), 2) AS avg_temp_diff
    FROM machines
    GROUP BY failure
""").show()

Failure rate by machine type:
+------------+--------------+--------+----------------+
|machine_type|total_machines|failures|failure_rate_pct|
+------------+--------------+--------+----------------+
|           L|          6000|     235|            3.92|
|           M|          2997|      83|            2.77|
|           H|          1003|      21|            2.09|
+------------+--------------+--------+----------------+

Average readings for failed vs working machines:
+-------+-------------+----------+---------+-------------+
|failure|avg_tool_wear|avg_torque|avg_speed|avg_temp_diff|
+-------+-------------+----------+---------+-------------+
|      1|        143.8|      50.2|   1496.5|          9.4|
|      0|        106.7|      39.6|   1540.3|        10.02|
+-------+-------------+----------+---------+-------------+



In [6]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, BinaryClassificationEvaluator
)

indexer = StringIndexer(inputCol="machine_type", outputCol="type_index")
df_model = indexer.fit(df).transform(df)

feature_cols = ["type_index", "air_temp", "process_temp",
                "rot_speed", "torque", "tool_wear", "temp_diff"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_model = assembler.transform(df_model).select("features", "failure")

train, test = df_model.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(featuresCol="features", labelCol="failure",
                            numTrees=50, seed=42)
model = rf.fit(train)
predictions = model.transform(test)

In [10]:
accuracy = MulticlassClassificationEvaluator(
    labelCol="failure", metricName="accuracy").evaluate(predictions)
auc = BinaryClassificationEvaluator(
    labelCol="failure", metricName="areaUnderROC").evaluate(predictions)
print(f"Accuracy: {accuracy:.3f}")
print(f"AUC: {auc:.3f}")

actual_failures = predictions.filter("failure = 1").count()
caught = predictions.filter("failure = 1 AND prediction = 1").count()
print(f"Real failures in test set: {actual_failures}")
print(f"Failures the model caught: {caught}")

for name, score in sorted(zip(feature_cols, model.featureImportances.toArray()),
                          key=lambda x: -x[1]):
    print(f"{name}: {score:.3f}")

Accuracy: 0.977
AUC: 0.921
Real failures in test set: 70
Failures the model caught: 26
rot_speed: 0.263
torque: 0.249
temp_diff: 0.235
tool_wear: 0.154
air_temp: 0.059
process_temp: 0.024
type_index: 0.016


In [14]:
# How much more a failure counts than a normal machine. Try 5, 10, or 20.
failure_weight = 5

train_w = train.withColumn(
    "weight", F.when(F.col("failure") == 1, float(failure_weight)).otherwise(1.0)
)

rf_w = RandomForestClassifier(featuresCol="features", labelCol="failure",
                              weightCol="weight", numTrees=50, seed=42)
model_w = rf_w.fit(train_w)
pred_w = model_w.transform(test)

accuracy_w = MulticlassClassificationEvaluator(
    labelCol="failure", metricName="accuracy").evaluate(pred_w)
auc_w = BinaryClassificationEvaluator(
    labelCol="failure", metricName="areaUnderROC").evaluate(pred_w)

actual = pred_w.filter("failure = 1").count()
caught = pred_w.filter("failure = 1 AND prediction = 1").count()
false_alarms = pred_w.filter("failure = 0 AND prediction = 1").count()

print(f"Weight used: {failure_weight}")
print(f"Accuracy: {accuracy_w:.3f}")
print(f"AUC: {auc_w:.3f}")
print(f"Real failures in test set: {actual}")
print(f"Failures the model caught: {caught}")
print(f"False alarms (flagged but fine): {false_alarms}")

Weight used: 5
Accuracy: 0.979
AUC: 0.940
Real failures in test set: 70
Failures the model caught: 42
False alarms (flagged but fine): 12
